# PDF Parser — KDIGO + NICE (unified)

One parser, one config per document. Runs the same 4-step pipeline on both PDFs and writes two JSON files.

**Inputs**
- `corpus/raw_pdfs/KDIGO_2024_CKD_Guideline_full.pdf` (199 pages)
- `corpus/raw_pdfs/NICE_NG203_CKD_assessment_and_management.pdf` (78 pages)

**Outputs**
- `corpus/parsed/kdigo_parsed.json`
- `corpus/parsed/nice_parsed.json`

**Per page:** `{document_name, source_url, page_number, section_title, text}`

**Parser:** PyMuPDF (tested against pdfplumber on the same page; pdfplumber glues words together on KDIGO's dense two-column layout).

In [1]:
import fitz  # PyMuPDF
import json
import re
import os

OUTPUT_DIR = "corpus/parsed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Configs — everything document-specific lives here

Each doc's config contains:
- `pdf_path` — where the file lives
- `document_name`, `source_url` — go into every page's metadata
- `content_pages` — (start, end) inclusive, 1-indexed — skip covers/TOC/references
- `noise_patterns` — regexes for lines to drop (headers, footers, page numbers)
- `section_patterns` — regexes for heading detection; first capture group = the title
- `chapter_overrides` — `{page_num: title}` for multi-line chapter titles we can't reliably regex
- `section_blacklist` — strings to reject as false-positive section matches
- `initial_section` — starting section label before any heading is found
- `output_file` — where to write the parsed JSON

In [2]:
KDIGO_CONFIG = {
    "pdf_path": "corpus/raw_pdfs/KDIGO_2024_CKD_Guideline_full.pdf",
    "document_name": "KDIGO 2024 CKD Guideline",
    "source_url": "https://kdigo.org/wp-content/uploads/2024/03/KDIGO-2024-CKD-Guideline.pdf",
    "content_pages": (14, 179),  # skip cover/TOC (1-13) and references (180-199)
    "noise_patterns": [
        re.compile(r'^\s*www\.kidney-international\.org\s*$'),
        re.compile(r'^\s*Kidney International \(2024\).*$'),
        re.compile(r'^\s*S\d{3,4}\s*$'),                                       # bare page labels like S160
        re.compile(r'^\s*summary of recommendation statements and practice points\s*$', re.IGNORECASE),
    ],
    "section_patterns": [
        re.compile(r'^(Chapter\s+\d+\s*[:–—].+)$'),                             # full chapter heading on one line
        re.compile(r'^([1-6]\.\d+(?:\.\d+)?\s+[A-Z][a-z].{5,})$'),              # "3.9 Glucagon-like...", not "1.73 m2"
        re.compile(r'^(Foreword|Notice|Acknowledgments|Summary of Changes)\s*$', re.IGNORECASE),
    ],
    "chapter_overrides": {
        54: "Chapter 1: Evaluation of CKD",
        81: "Chapter 2: Risk assessment in people with CKD",
        90: "Chapter 3: Delaying CKD progression and managing its complications",
        131: "Chapter 4: Medication management and drug stewardship in CKD",
        140: "Chapter 5: Optimal models of care",
        155: "Chapter 6: Research recommendations",
    },
    "section_blacklist": {"4.0 International License"},
    "initial_section": "Summary of Recommendations",
    "output_file": "kdigo_parsed.json",
}

NICE_CONFIG = {
    "pdf_path": "corpus/raw_pdfs/NICE_NG203_CKD_assessment_and_management.pdf",
    "document_name": "NICE NG203 - Chronic kidney disease: assessment and management",
    "source_url": "https://www.nice.org.uk/guidance/ng203",
    "content_pages": (5, 78),  # skip cover/responsibility/TOC (1-4)
    "noise_patterns": [
        re.compile(r'^\s*Chronic kidney disease: assessment and management.*NICE guideline.*$', re.IGNORECASE),
        re.compile(r'^\s*©\s*NICE\s*20\d\d.*$'),                                # copyright footer
        re.compile(r'^\s*conditions#notice-of-rights\)\.\s*$'),                    # tail of that footer
        re.compile(r'^\s*www\.nice\.org\.uk/guidance/ng203\s*$'),                  # URL footer
        re.compile(r'^\s*Page\s+\d+\s+of\s*$'),                                    # "Page X of"
        re.compile(r'^\s*\d{1,3}\s*$'),                                            # bare page number (the "78" that follows)
    ],
    "section_patterns": [
        # "1.6 Pharmacotherapy..." (top-level numbered sections)
        re.compile(r'^(1\.\d{1,2}\s+[A-Z][a-z].{5,})$'),
        # Standalone headings
        re.compile(r'^(Overview|Recommendations|Terms used in this guideline|Context|Finding more information|Update information|Your responsibility)\s*$'),
    ],
    "chapter_overrides": {},   # NICE has flat 1.x structure, no multi-line chapter titles
    "section_blacklist": set(),
    "initial_section": "Preamble",
    "output_file": "nice_parsed.json",
}

CONFIGS = [KDIGO_CONFIG, NICE_CONFIG]
print(f"Configured {len(CONFIGS)} documents to parse.")

Configured 2 documents to parse.


## The pipeline (identical for every document)

1. **Extract** — PyMuPDF, page by page, preserving `page_number`
2. **Clean** — drop noise-pattern lines, remove `(cid:xx)` artifacts, fix broken hyphenation, collapse blank lines
3. **Detect sections** — apply this doc's regex patterns + chapter overrides; carry forward the most recent heading
4. **Save** — one JSON file per doc

In [3]:
CID_PATTERN = re.compile(r'\(cid:\d+\)')


def extract_pages(cfg):
    """Step 1: raw text per page."""
    doc = fitz.open(cfg["pdf_path"])
    start, end = cfg["content_pages"]
    pages = []
    for pg_num in range(start - 1, end):  # 0-indexed range
        pages.append({
            "document_name": cfg["document_name"],
            "source_url": cfg["source_url"],
            "page_number": pg_num + 1,
            "raw_text": doc[pg_num].get_text(),
        })
    doc.close()
    return pages


def clean_page(raw_text, noise_patterns):
    """Step 2: drop noise lines, fix artifacts and hyphenation."""
    cleaned = []
    for line in raw_text.split('\n'):
        if any(p.match(line) for p in noise_patterns):
            continue
        cleaned.append(CID_PATTERN.sub('', line))
    text = '\n'.join(cleaned)
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)      # "concen-\ntration" -> "concentration"
    text = re.sub(r'\n{3,}', '\n\n', text)             # collapse blank runs
    return text.strip()


def detect_sections(pages, cfg):
    """Step 3: tag each page with the most-recent section heading."""
    current = cfg["initial_section"]
    for page in pages:
        # Chapter overrides win (they anchor multi-line chapter titles reliably)
        if page["page_number"] in cfg["chapter_overrides"]:
            current = cfg["chapter_overrides"][page["page_number"]]
        for line in page["text"].split('\n'):
            s = line.strip()
            for pat in cfg["section_patterns"]:
                m = pat.match(s)
                if m:
                    candidate = m.group(1).strip()
                    if not any(b in candidate for b in cfg["section_blacklist"]):
                        current = candidate
                    break
        page["section_title"] = current
    return pages


def parse_document(cfg):
    """Full pipeline for one document. Returns cleaned+tagged pages."""
    raw = extract_pages(cfg)
    cleaned = [{
        "document_name": p["document_name"],
        "source_url": p["source_url"],
        "page_number": p["page_number"],
        "text": clean_page(p["raw_text"], cfg["noise_patterns"]),
    } for p in raw]
    return detect_sections(cleaned, cfg), raw   # return raw too for inspection


print("Pipeline functions defined.")

Pipeline functions defined.


## Run the parser on both documents

In [4]:
results = {}   # {document_name: (cleaned_pages, raw_pages, cfg)}

for cfg in CONFIGS:
    print(f"\n>>> Parsing: {cfg['document_name']}")
    cleaned, raw = parse_document(cfg)

    # Save output
    out_path = os.path.join(OUTPUT_DIR, cfg["output_file"])
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(cleaned, f, indent=2, ensure_ascii=False)

    # Stats
    total_chars = sum(len(p['text']) for p in cleaned)
    empty = sum(1 for p in cleaned if len(p['text'].strip()) < 50)
    unique_sections = len({p['section_title'] for p in cleaned})

    print(f"    Pages parsed: {len(cleaned)}")
    print(f"    Total chars: {total_chars:,}")
    print(f"    Near-empty pages (<50 chars): {empty}")
    print(f"    Unique sections detected: {unique_sections}")
    print(f"    Saved to: {out_path} ({os.path.getsize(out_path)/1024:.1f} KB)")

    results[cfg["document_name"]] = (cleaned, raw, cfg)


>>> Parsing: KDIGO 2024 CKD Guideline


    Pages parsed: 166
    Total chars: 752,712
    Near-empty pages (<50 chars): 0
    Unique sections detected: 54
    Saved to: corpus/parsed\kdigo_parsed.json (803.2 KB)

>>> Parsing: NICE NG203 - Chronic kidney disease: assessment and management
    Pages parsed: 74
    Total chars: 117,458
    Near-empty pages (<50 chars): 0
    Unique sections detected: 16
    Saved to: corpus/parsed\nice_parsed.json (136.2 KB)


## Section maps — visually verify the section detection for each doc

In [5]:
for name, (cleaned, _, _) in results.items():
    print(f"\n===== {name} =====")
    seen = []
    for p in cleaned:
        if p['section_title'] not in seen:
            seen.append(p['section_title'])
            print(f"  p{p['page_number']:>3}: {p['section_title'][:90]}")
    print(f"  → {len(seen)} sections")


===== KDIGO 2024 CKD Guideline =====
  p 14: notice
  p 15: foreword
  p 34: 1.1.4 Evaluation of cause
  p 35: 1.2.1 Other functions of kidneys besides GFR
  p 36: 1.2.2 Guidance to physicians and other healthcare providers
  p 38: 1.2.3 Guidance to clinical laboratories
  p 39: 1.3.1 Guidance for physicians and other healthcare providers
  p 40: 2.1 Overview on monitoring for progression of CKD based upon GFR and ACR categories
  p 41: 3.2.1 Avoiding use of tobacco products
  p 42: 3.3.2 Sodium intake
  p 43: 3.6 Renin-angiotensin system inhibitors
  p 44: 3.8 Mineralocorticoid receptor antagonists (MRA)
  p 45: 3.11.3 Timing to recheck potassium after identifying moderate and severe hyperkalemia in a
  p 46: 3.15.1 Lipid management
  p 47: 3.15.3 Invasive versus intensive medical therapy for coronary artery disease
  p 49: 4.2 Dose adjustments by level of GFR
  p 50: 4.4.2 Gadolinium-containing contrast media
  p 51: 5.2.2 Identiﬁcation and assessment of symptoms
  p 52: 5.3.1 Trans

## Cleaning inspection — diff raw vs cleaned on one page per doc

Confirm we removed only noise and kept clinical content.

In [6]:
SAMPLE_PAGES = {
    "KDIGO 2024 CKD Guideline": 45,
    "NICE NG203 - Chronic kidney disease: assessment and management": 24,
}

for name, (cleaned, raw, cfg) in results.items():
    pg = SAMPLE_PAGES[name]
    start = cfg["content_pages"][0]
    idx = pg - start
    raw_lines = set(raw[idx]['raw_text'].strip().split('\n'))
    clean_lines = set(cleaned[idx]['text'].strip().split('\n'))
    removed = sorted(r for r in (raw_lines - clean_lines) if r.strip())

    print(f"\n===== {name} — PAGE {pg} =====")
    print(f"raw: {len(raw[idx]['raw_text'])} chars  |  cleaned: {len(cleaned[idx]['text'])} chars")
    print(f"Lines removed ({len(removed)}):")
    for r in removed[:10]:
        print(f"  [-] {r[:100]}")


===== KDIGO 2024 CKD Guideline — PAGE 45 =====
raw: 4304 chars  |  cleaned: 4150 chars
Lines removed (12):
  [-] (ﬁnerenone). Adapted from the protocols of Finerenone in Reducing Kidney Failure and Disease Progres
  [-] DKD) and Finerenone in Reducing Cardiovascular Mortality and Morbidity in Diabetic Kidney Disease (F
  [-] Kidney International (2024) 105 (Suppl 4S), S117–S314
  [-] Practice Point 3.11.1.1: Be aware of the variability of potassium laboratory measurements as well as
  [-] Practice Point 3.11.2.1: Be aware of local availability or formulary restrictions with regard to the
  [-] Practice Point 3.8.3: To mitigate risk of hyperkalemia, select people with consistently normal serum
  [-] S160
  [-] agement of nonemergent hyperkalemia.
  [-] anisms that may inﬂuence potassium measurement including diurnal and seasonal variation,
  [-] summary of recommendation statements and practice points

===== NICE NG203 - Chronic kidney disease: assessment and management — PAGE 24 =====

## Spot check — trace one page per doc back to the PDF

Deck requirement: *"Pick one chunk and trace it back to the original PDF page."*

In [7]:
for name, (cleaned, _, cfg) in results.items():
    pg = SAMPLE_PAGES[name]
    idx = pg - cfg["content_pages"][0]
    p = cleaned[idx]
    print(f"\n===== {name} =====")
    print(f"page_number:   {p['page_number']}")
    print(f"section_title: {p['section_title']}")
    print(f"text (first 400 chars):")
    print(p['text'][:400])
    print(f"... ({len(p['text'])} chars total)")


===== KDIGO 2024 CKD Guideline =====
page_number:   45
section_title: 3.11.3 Timing to recheck potassium after identifying moderate and severe hyperkalemia in adults
text (first 400 chars):
Practice Point 3.8.2: A nonsteroidal MRA may be added to a RASi and an SGLT2i for treatment of T2D and CKD in adults.
Practice Point 3.8.3: To mitigate risk of hyperkalemia, select people with consistently normal serum potassium concentration and monitor serum potassium regularly after initiation of a nonsteroidal MRA (Figure 26).
Practice Point 3.8.4: The choice of a nonsteroidal MRA should prior
... (4150 chars total)

===== NICE NG203 - Chronic kidney disease: assessment and management =====
page_number:   24
section_title: 1.6 Pharmacotherapy
text (first 400 chars):
1.6.1 
In adults with CKD and an ACR under 70 mg/mmol, aim for a clinic systolic blood 
pressure below 140 mmHg (target range 120 to 139 mmHg) and a clinic diastolic 
blood pressure below 90 mmHg. [2021] 
1.6.2 
In adults with CKD a

## Summary

Two JSON files, same schema, ready for the next step (section-aware chunking):
```json
{"document_name": "...", "source_url": "...", "page_number": 45,
 "section_title": "3.9 Glucagon-like peptide-1 receptor agonists", "text": "..."}
```

**Adding a third document later** = add one more config dict to `CONFIGS`. No changes to the pipeline.